# M3C-001-TPU-JAX
选择 TPU、开启 Internet，沿用 GITHUB_TOKEN。Save & Run All。先 SISO 再 MIMO；仅合成正确性检查，0 参数更新。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'research/m3c-001-tpu-jax'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-m3c-tpu-jax'
OUTPUT = WORKSPACE / 'm3c-tpu-jax-bootstrap'
HARD_LIMIT_SECONDS = 7200
started = time.monotonic()
PLATFORM_RESERVE_SECONDS = 60
deadline = started + HARD_LIMIT_SECONDS - PLATFORM_RESERVE_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('m3c-tpu-jax-' + SOURCE_COMMIT[:12])

CANDIDATE = WORKSPACE / 'm3c-jax-candidate'
PIN = 'ffa55848fc513ca91aa330bda4c33c86dab74827'
if not CANDIDATE.exists():
    run_bounded(['git','clone','--depth=1','--no-checkout','https://github.com/jax-state-spaces/mamba3-jax.git',str(CANDIDATE)],timeout_seconds=180)
run_bounded(['git','-C',str(CANDIDATE),'fetch','--depth=1','origin',PIN],timeout_seconds=120)
run_bounded(['git','-C',str(CANDIDATE),'checkout','--detach',PIN])
# Isolated interpreter avoids imported JAX/Flax from the notebook process.
ENV = WORKSPACE / 'm3c-jax-env'
if not (ENV / 'bin/python').exists():
    run_bounded([sys.executable,'-m','venv','--system-site-packages',str(ENV)],timeout_seconds=120)
PYTHON = str(ENV / 'bin/python')
run_bounded([PYTHON,'-m','pip','install','jax[tpu]>=0.8.0','flax>=0.12.0','pytest','absl-py','-e',str(CANDIDATE)],timeout_seconds=600)
run_bounded([PYTHON,str(REPO / 'research/projects/cellular-language-models/experiments/mamba3-cellular-series/M3C-001-TPU-JAX/run_audit.py'),'--output',str(OUTPUT),'--candidate',str(CANDIDATE)],cwd=REPO,timeout_seconds=1800)
